In [0]:
%sql

INSERT OVERWRITE proyecto_final.silver.barrios (
    barrio_id,
    barrio_nombre,
    tipo_entidad,
    comuna_id,
    perimetro_m,
    area_m2,
    coordenadas
)
WITH datos_limpios AS (
    SELECT 
    id AS barrio_id,
    LOWER(REPLACE(TRIM(nombre), '"', '')) AS barrio_nombre,
    LOWER(REPLACE(TRIM(objeto), '"', '')) AS tipo_entidad,
    comuna AS comuna_id,
    ROUND(perimetro_, 2) AS perimetro_m,
    ROUND(area_metro, 2) AS area_m2,
    geometry AS coordenadas
    FROM proyecto_final.raw.barrios_bronze
    WHERE nombre IS NOT NULL 
      AND geometry IS NOT NULL
),
datos_deduplicados AS (
  SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY 
                barrio_id, 
                barrio_nombre
            ORDER BY barrio_id 
        ) AS rn
  from datos_limpios
)
SELECT 
    barrio_id,
    barrio_nombre,
    tipo_entidad,
    comuna_id,
    perimetro_m,
    area_m2,
    coordenadas
FROM datos_deduplicados
WHERE rn = 1;

In [0]:
select * from proyecto_final.silver.barrios